# Statistical Language Modelling with KenLM


N-gram language models trained on the complete works of Shakespeare with [KenLM](https://github.com/kpu/kenlm): building an ARPA model with `lmplz`, measuring perplexity on held-out text, and generating text across n-gram orders and decoding strategies.

Coursework for ICS472 (Natural Language Processing), KFUPM. Run on Google Colab, where KenLM is built from source below.


### Installing and building KenLM

KenLM is a C++ toolkit, so it needs compiling from source along with its Boost dependencies. Paths below are Colab's.


In [1]:
!pip install https://github.com/kpu/kenlm/archive/master.zip --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.6/553.6 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!git clone https://github.com/kpu/kenlm.git --quiet

In [8]:
!apt-get update -qq
!apt-get install -y -qq build-essential cmake libboost-program-options-dev libboost-system-dev libboost-thread-dev zlib1g-dev libbz2-dev liblzma-dev libeigen3-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libboost-atomic1.74.0:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../00-libboost-atomic1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-atomic1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-atomic1.74-dev:amd64.
Preparing to unpack .../01-libboost-atomic1.74-dev_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-atomic1.74-dev:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-chrono1.74.0:amd64.
Preparing to unpack .../02-libboost-chrono1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-chrono1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-chrono1.74-dev:amd64.
Preparing to unpack .../03-libb

In [26]:
%cd /content/kenlm
!pwd
!ls

/content/kenlm
/content/kenlm
build		     compile_query_only.sh  LICENSE	    README.md
BUILDING	     COPYING		    lm		    setup.py
clean_query_only.sh  COPYING.3		    MANIFEST.in     util
cmake		     COPYING.LESSER.3	    pyproject.toml
CMakeLists.txt	     Doxyfile		    python


In [16]:
!apt-get update -qq
!apt-get install -y -qq libboost-test-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libboost-test1.74.0:amd64.
(Reading database ... 118954 files and directories currently installed.)
Preparing to unpack .../libboost-test1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-test1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-test1.74-dev:amd64.
Preparing to unpack .../libboost-test1.74-dev_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-test1.74-dev:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-test-dev:amd64.
Preparing to unpack .../libboost-test-dev_1.74.0.3ubuntu7_amd64.deb ...
Unpacking libboost-test-dev:amd64 (1.74.0.3ubuntu7) ...
Setting up libboost-test1.74.0:amd64 (1.74.0-14ubuntu3) ...
Setting up libboost-test1.74-dev:amd64 (1.74.0-14ubuntu3) ...
Setting up libb

In [27]:
!apt-get update -qq
!apt-get install -y -qq libboost-all-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package javascript-common.
(Reading database ... 118988 files and directories currently installed.)
Preparing to unpack .../00-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libboost1.74-tools-dev.
Preparing to unpack .../01-libboost1.74-tools-dev_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost1.74-tools-dev (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-tools-dev.
Preparing to unpack .../02-libboost-tools-dev_1.74.0.3ubuntu7_amd64.deb ...
Unpacking libboost-tools-dev (1.74.0.3ubuntu7) ...
Selecting previously unselected package libboost-atomic-dev:amd64.
Preparing to unpack .../03-libboost-atomic-dev_1.74.0.3ubuntu7_amd64.deb ...

In [28]:
%cd /content/kenlm
!rm -rf build
!mkdir build
%cd /content/kenlm/build
!cmake ..
!make -j4
!ls /content/kenlm/build/bin

/content/kenlm
/content/kenlm/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

-- Found Boost: /usr/lib/x86_64-linux-gnu/cmake/Boost-1.74.0/BoostConfig.cmake (found suitable version "1.74.0", minimum required is 

In [29]:
!ls /content/kenlm/build/bin

build_binary  fragment	       lmplz			     query
count_ngrams  interpolate      phrase_table_vocab	     streaming_example
filter	      kenlm_benchmark  probing_hash_table_benchmark


In [51]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set(style='darkgrid', font_scale=1.6)
import requests
import nltk
from nltk.corpus import treebank, brown, conll2000
import os
import subprocess
import kenlm
import random

In [52]:
# Load text file
text = open('/content/text.txt', 'r').read()

# Preview text
print(text[:300])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [53]:
# Split text into lines
lines = text.split('\n')

# Define split ratios
train_ratio = 0.8
test_ratio = 0.2

# Calculate split indices based on the number of lines
total_lines = len(lines)
train_end = int(total_lines * train_ratio)

In [54]:
# Split the lines into train, validation, and test sets
train_lines = lines[:train_end]
test_lines = lines[train_end:]

# Join the lines back into text
train_text = '\n'.join(train_lines)
test_text = '\n'.join(test_lines)

In [55]:
# Check splits
print(f"Train length: {len(train_text)}")
print(f"Test length: {len(test_text)}")

Train length: 907167
Test length: 208225


In [56]:
# Preview the first 300 characters of each split
print("\nTrain Text Preview:\n", train_text[:300])
print("\nTest Text Preview:\n", test_text[:300])


Train Text Preview:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us

Test Text Preview:
 Nay, if there be no remedy for it, but that you will
needs buy and sell men and women like beasts, we
shall have all the world drink brown and white bastard.

DUKE VINCENTIO:
O heavens! what stuff is here

POMPEY:
'Twas never merry world since, of two usuries, the
merriest was put down, and the wors


In [57]:
%cd /content
!pwd

/content
/content


In [58]:
# Save each split to a separate file for KenLM
#with open('train.txt', 'w', encoding='utf-8') as f:
    #f.write(train_text)

#with open('test.txt', 'w', encoding='utf-8') as f:
    #f.write(test_text)

# Save each split to a separate file for KenLM
with open('/content/train.txt', 'w', encoding='utf-8') as f:
    f.write(train_text)

with open('/content/test.txt', 'w', encoding='utf-8') as f:
    f.write(test_text)

### Paths

Absolute Colab paths. The commented alternatives are the relative equivalents for a local run.


In [59]:
#arpa_model_path = 'kenlm/kenlm_model.arpa'
#binary_model_path = 'kenlm/kenlm_model.bin'
#kenlm_bin_path = 'kenlm/build/bin/'
#corpus_path = 'train.txt'

#Using Google Collab
arpa_model_path = '/content/kenlm/kenlm_model.arpa'
binary_model_path = '/content/kenlm/kenlm_model.bin'
kenlm_bin_path = '/content/kenlm/build/bin/'
corpus_path = '/content/train.txt'

## Building the ARPA model

`lmplz` estimates an n-gram model with modified Kneser-Ney smoothing. Order 5 here, trained on the 80% training split.


In [60]:
result = subprocess.run(
    [f'{kenlm_bin_path}lmplz', '-o', '5', '--text', corpus_path, '--arpa', arpa_model_path],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

In [61]:
print("Q1 return code:", result.returncode)
print("STDERR:\n", result.stderr.decode('utf-8'))
print("ARPA exists:", os.path.exists('/content/kenlm/kenlm_model.arpa'))
print("Train exists:", os.path.exists('/content/train.txt'))
!ls -lh /content/train.txt
!ls -lh /content/kenlm/kenlm_model.arpa

Q1 return code: 0
STDERR:
 === 1/5 Counting and sorting n-grams ===
Reading /content/train.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 164806 types 22329
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:267948 2:1061892672 3:1991048832 4:3185678080 5:4645780480
Statistics:
1 22329 D1=0.690029 D2=1.04297 D3+=1.49191
2 100400 D1=0.841654 D2=1.12861 D3+=1.43849
3 140565 D1=0.938327 D2=1.28841 D3+=1.42571
4 133159 D1=0.980379 D2=1.50124 D3+=1.90971
5 114859 D1=0.992693 D2=1.79486 D3+=2.17846
Memory estimate for binary LM:
type       kB
probing 11354 assuming -p 1.5
probing 13633 assuming -r models -p 1.5
trie     5536 without quantization
trie     3074 assuming -q 8 -b 8 quantization 
trie     5048 assuming -a 22 array pointer compression
trie     2587 assuming -a 22 -q 8 -b 8 array pointer compres

In [62]:
# Convert ARPA to binary format (optional, for faster querying)
result = subprocess.run([f'{kenlm_bin_path}build_binary', arpa_model_path, binary_model_path],
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE)

In [63]:
def display_ngram_samples(arpa_model_path, max_n=5, sample_size=5):
    """
    Display the first few learned n-grams for each n-gram level from a KenLM ARPA model.

    :param arpa_model_path: Path to the ARPA model file.
    :param max_n: Maximum n-gram level to display (default is 5).
    :param sample_size: Number of samples to show per n-gram level (default is 5).
    """

    # Define the n-gram sections to look for in the ARPA file
    n_gram_sections = {n: f'\\{n}-grams:' for n in range(1, max_n + 1)}

    # Store the first 'sample_size' n-grams for each level
    n_gram_samples = {n: [] for n in n_gram_sections.keys()}

    current_n_gram = None

    # Read and parse the ARPA model file
    with open(arpa_model_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # Detect the start of an n-gram section
            for n, section in n_gram_sections.items():
                if line.startswith(section):
                    current_n_gram = n
                    break

            # Collect n-gram data if within a valid section
            if current_n_gram and not line.startswith(("\\", "ngram", "data")):
                if len(n_gram_samples[current_n_gram]) < sample_size:
                    n_gram_samples[current_n_gram].append(line)

    # Display the results
    for n in range(1, max_n + 1):
        print(f"\nFirst {sample_size} {n}-grams learned by the model:")
        if n_gram_samples[n]:
            for sample in n_gram_samples[n]:
                print(sample)
        else:
            print(f"No {n}-gram data found in the ARPA model.")

In [64]:
display_ngram_samples(arpa_model_path, max_n=5, sample_size=5)


First 5 1-grams learned by the model:
-5.0374475	<unk>	0
0	<s>	-0.986783
-1.0221202	</s>	0
-4.728018	First	-0.07486659
-4.130795	Citizen:	-0.7451817

First 5 2-grams learned by the model:
-0.7320903	<s> </s>	0
-0.07713112	Citizen: </s>	0
-1.2938557	we </s>	0
-0.5401436	proceed </s>	0
-1.141414	any </s>	0

First 5 3-grams learned by the model:
-0.034954496	<s> Citizen: </s>	0
-0.07195657	First Citizen: </s>	0
-0.07195657	Second Citizen: </s>	0
-0.07195657	Third Citizen: </s>	0
-0.07195657	Fourth Citizen: </s>	0

First 5 4-grams learned by the model:
-0.0029550774	<s> First Citizen: </s>	0
-0.0053087818	<s> Second Citizen: </s>	0
-0.005794646	<s> Third Citizen: </s>	0
-0.04440686	<s> Fourth Citizen: </s>	0
-0.07042374	<s> Fifth Citizen: </s>	0

First 5 5-grams learned by the model:
-0.98625463	they shall know we </s>
-0.8754098	And to poor we </s>
-1.141983	young prince as we </s>
-0.98625463	Why, what need we </s>
-0.45582402	not Marcius, we'll proceed </s>


## Loading the compiled model

The binary format loads considerably faster than the ARPA text format.


In [65]:
model = kenlm.Model(binary_model_path)

## Perplexity

Perplexity on the held-out test split. Lower is better: it is the exponentiated average negative log-likelihood per token, so roughly the number of equally likely words the model is choosing between at each step.


In [66]:
# Function to calculate perplexity of a file using KenLM
def calculate_perplexity(file_path, model):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    total_log_prob = 0.0
    total_words = 0

    for line in lines:
        line = line.strip()
        if not line:
            continue

        total_log_prob += model.score(line, bos=True, eos=True)
        total_words += len(line.split()) + 1   # +1 for </s>

    perplexity = 10 ** (-total_log_prob / total_words)

    return perplexity

In [70]:
# Calculate perplexity on the test set
test_perplexity = calculate_perplexity('test.txt', model)

In [71]:
print(f"Test Perplexity: {test_perplexity}")

Test Perplexity: 634.6546379245973


## Generating text

At each step the model scores candidate continuations given the previous `n_gram - 1` tokens and picks from the `top_k` highest-scoring. `top_k = 1` is greedy decoding; larger values sample among the best candidates.


In [72]:
def generate_text(model, seed_text, num_words=20, n_gram=5, top_k=5):
    """
    Generate text using a KenLM model with randomness in selection.

    :param model: The loaded KenLM model.
    :param seed_text: Initial seed text to start the generation.
    :param num_words: Number of words to generate.
    :param n_gram: Maximum number of words to consider for context (n-gram size).
    :param top_k: Select from the top K highest-scoring words to introduce randomness.
    :return: Generated text as a string.
    """

    with open('/content/train.txt', 'r', encoding='utf-8') as f:
        vocabulary = sorted(set(f.read().split()))

    words = seed_text.split()

    for _ in range(num_words):
        context = words[-(n_gram - 1):] if n_gram > 1 else []
        scored_candidates = []

        for candidate in vocabulary:
            candidate_text = " ".join(context + [candidate])
            score = model.score(candidate_text, bos=False, eos=False)
            scored_candidates.append((candidate, score))

        scored_candidates.sort(key=lambda x: x[1], reverse=True)
        k = min(top_k, len(scored_candidates))
        next_word = random.choice(scored_candidates[:k])[0]
        words.append(next_word)

    text = " ".join(words)

    return text

## Generation across n-gram orders and decoding strategies

The same seed with every combination of n-gram order (5, 4, 3, 2) and `top_k` (1, 5).

The seed is **"Romeo took a plane to visit his uncle"** — deliberately not Shakespearean. The word *plane* never appears in the corpus, and neither does the phrase *visit his uncle*.


**5-grams, highest-scoring word**


In [73]:

n_gram= 5
top_k= 1

In [74]:
generated_text = generate_text(model, "Romeo took a plane to visit his uncle", num_words=20, n_gram=n_gram, top_k= top_k)
print(generated_text)

Romeo took a plane to visit his uncle Gaunt a father, for the world is the way to make me to the king in the world is the


**5-grams, top 5 scoring words**


In [75]:

n_gram= 5
top_k= 5

In [76]:
generated_text = generate_text(model, "Romeo took a plane to visit his uncle", num_words=20, n_gram=n_gram, top_k= top_k)
print(generated_text)

Romeo took a plane to visit his uncle with the other to his charge: to your and I must not say the and to give me to your


**4-grams, highest-scoring word**


In [88]:

n_gram= 4
top_k= 1

In [89]:
generated_text = generate_text(model, "Romeo took a plane to visit his uncle", num_words=20, n_gram=n_gram, top_k= top_k)
print(generated_text)

Romeo took a plane to visit his uncle Gaunt a father, for the world is the way to make me to the king in the world is the


**4-grams, top 5 scoring words**


In [78]:

n_gram= 4
top_k= 5

In [79]:
generated_text = generate_text(model, "Romeo took a plane to visit his uncle", num_words=20, n_gram=n_gram, top_k= top_k)
print(generated_text)

Romeo took a plane to visit his uncle is to the and to make his prey. to my lord, as the and my young prince as the air


**Tri-grams, highest-scoring word**


In [80]:

n_gram= 3
top_k= 1

In [81]:
generated_text = generate_text(model, "Romeo took a plane to visit his uncle", num_words=20, n_gram=n_gram, top_k= top_k)
print(generated_text)

Romeo took a plane to visit his uncle Gaunt a father, for the world is the way to make me to the king and not the king and


**Tri-grams, top 5 scoring words**


In [82]:

n_gram= 3
top_k= 5

In [83]:
generated_text = generate_text(model, "Romeo took a plane to visit his uncle", num_words=20, n_gram=n_gram, top_k= top_k)
print(generated_text)

Romeo took a plane to visit his uncle is the way to his father's death. But who is my lord. My suit of your royal and the rest,


**Bi-grams, highest-scoring word**


In [84]:

n_gram= 2
top_k= 1

In [85]:
generated_text = generate_text(model, "Romeo took a plane to visit his uncle", num_words=20, n_gram=n_gram, top_k= top_k)
print(generated_text)

Romeo took a plane to visit his uncle York and the king and the king and the king and the king and the king and the king and


**Bi-grams, top 5 scoring words**


In [86]:

n_gram= 2
top_k= 5

In [87]:
generated_text = generate_text(model, "Romeo took a plane to visit his uncle", num_words=20, n_gram=n_gram, top_k= top_k)
print(generated_text)

Romeo took a plane to visit his uncle is a thousand lives shall have a to my heart is not to his son I am to be a


## Observations


From the generated examples, I noticed that increasing the n-gram size generally produced more fluent and locally coherent text, while smaller n-grams made the output more repetitive and less natural. The 5-gram and 4-gram settings gave better sentence flow because they used a larger context window, whereas the bigram model often repeated the same patterns, such as looping phrases like “the king and the king.” I also observed that choosing the highest-scoring word (top_k = 1) made the output more deterministic, but it often became repetitive and predictable. In contrast, using the top 5 highest-scoring words (top_k = 5) introduced more variation and made the text sound slightly more creative, although it sometimes reduced grammatical consistency. Overall, the best balance in my results came from higher-order n-grams with top_k = 5, since they produced text that was more diverse while still remaining relatively coherent.
